#Log
* trim each column
* remove the first 3 letter of CID column to follow id from customer_info
* normalize GEN column to Male and Female
* change the Birth date on BDATE column that is beyond today date as none
* rename column

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length
     

In [0]:
cust_az12 = spark.table("workspace.bronze.erp_cust_az12")

cust_az12.display()

In [0]:
%sql
--VALIDATE if there is bdate that not in date format 
SELECT *
FROM workspace.bronze.erp_cust_az12
WHERE len(BDATE)<10

In [0]:
%sql
-- validate if there is future birthdate
-- there is 16 row with future birthdate
SELECT *
FROM workspace.bronze.erp_cust_az12
WHERE BDATE>current_date()

In [0]:
## null counts
## there is 1472 null IN GEN
null_counts_cust_az12=cust_az12.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cust_az12.columns])

null_counts_cust_az12.show()

In [0]:
%sql
-- validate format in GEN column
-- there 9 different values, expect two have male,female
SELECT DISTINCT GEN
FROM workspace.bronze.erp_cust_az12

In [0]:
# trim each column
for field in cust_az12.schema.fields:
    if isinstance(field.dataType, StringType):
        cust_az12= cust_az12.withColumn(field.name, trim(col(field.name)))

In [0]:
# remove the first 3 letter in CID
cust_az12 = cust_az12.withColumn(
    "CID",
    F.substring(col("CID"),4,F.length(col("CID"))-3)
                                 )
                                 
                                

In [0]:
cust_az12=cust_az12.withColumn(
    "GEN",
    F.when(F.upper(F.col("GEN")).isin("M","MALE"),"Male").
    when(F.upper(F.col("GEN")).isin("F","FEMALE"),"Female").
    otherwise("Unknown")
)

In [0]:
# validate gen 
cust_az12_gen_validate = cust_az12.select("GEN").distinct().show()

In [0]:
# change from future birthdate to none
cust_az12 = cust_az12.withColumn(
    "BDATE",
    F.when(col("BDATE")>F.current_date(),None).
    otherwise(F.to_date(col("BDATE")))
)

In [0]:
Rename_map_cust_az12 ={
    "CID":"customer_id",
    "BDATE":"birth_date",
    "GEN":"gender"
}
for old_name, new_name in Rename_map_cust_az12.items():
    cust_az12 = cust_az12.withColumnRenamed(old_name,new_name)

In [0]:
cust_az12.display()

In [0]:
cust_az12.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_cust_az12")

In [0]:
%sql
SELECT *
FROM workspace.silver.erp_cust_az12